# Look-ahead · Mechanism — where in the pipeline does K act?  `[TRAINING] → [EVAL]`

**Family `lookahead/mechanism`.** `lookahead/reward` reports *that* K=0 and K=5 end up at different rubric
scores and `lookahead/behaviour` reports *what* the conversations look like; this family asks **where** the
look-ahead lever acts, on the training-time record the trainers left behind (`iteration_N/eda/generations.jsonl`
— every candidate the policy sampled, the score the training oracle gave it, and for K=5 the five simulated turns
that were appended before scoring). No new oracle calls. Four questions:

- **§1 · The over-praise chain** — select → generate → evaluate for both PTO look-ahead arms: is K=5's flat
  over-praise visible at the *first* gate (what the reward selects for), or only at the outcome?
  (`k_mechanism_overpraise_chain`, `k_mechanism_overpraise`; ported from the retired `6_Preference` §5d.)
- **§2 · Dispersion** — does look-ahead *widen* the within-group training signal or merely *rescale* it?
  (`dispersion_by_iter`, `dispersion_ratios`, `dispersion_tau`, `dispersion_expectation`; figures `dispersion`,
  `dispersion_tau`.)
- **§3 · Reward faithfulness** — is the partial-conversation training proxy a faithful proxy for the
  full-conversation eval, does K change that, and does the answer survive a held-out grader?
  (`faithfulness_*` tables; figures `faithfulness`, `faithfulness_heldout`.)
- **§4 · The tail audit** — what the K=5 reward actually scored: how the simulated tails end, whether an early
  ending predicts the reward *within the group the update sees*. (`tail_audit_*` tables; figure `tail_audit`.)
- **§5 · Ledgers** — `dispersion_numbers.json`, `faithfulness_numbers.json`, `tails_numbers.json`.

> **Judge-invariant family.** §1 rows 1–2, §2 and §4 are training-side numbers: `generations.jsonl` records
> what the *training* oracle (gpt-4o-mini) scored during the run, and no later grader can change that. §3 puts
> the primary (training) oracle and the held-out judge (Claude Haiku 4.5) side by side on the EVAL side of the
> join. Exports carry no `<judge>/` level (`results/lookahead/mechanism/{figures,tables}/`); `EDA_JUDGE` is
> ignored. Promoted 2026-08-18 from `6_Preference` §5d and the paper generators
> `papers/2026_lookahead_pto_grpo/analysis/{dispersion_by_k,reward_faithfulness,tail_audit}.py` into
> `eda_analysis.{pref,dispersion,faithfulness,tails}`; the paper's frozen `tables/*.csv` are the fixture (means /
> dz / p / counts exact; every bootstrap CI is re-drawn under `constants.BOOT_SEED`, so CI bounds may differ from
> the fixture in the third decimal). The API-call accounting from `tail_audit.py` renders in `compute/cost`.

**Conventions (every table repeats them in its caption).** Sign of a cross-K contrast in §3: **+ ⇒ K=0 higher**
(delta = K0 − K5; for *faithfulness* a NEGATIVE delta means look-ahead helped). §2 uses ratios **K5 / K0** (>1 =
K=5 wider) and §4 within-group contrasts (**+ ⇒ ended-early candidates score higher**) — the caption of each
table names its own convention. Pairing unit: §1–§2, §4 = the *group* (the 8 candidates the policy sampled at
one branch point, keyed `(arm, train_iter, conversation_id, branch_id, epoch)` — PTO's `branch_id` is the trunk
depth, so `conversation_id` is required); §3 = the *conversation* (cluster bootstrap over conversations within a
model state). **GRPO K=5 is right-censored at iteration 5** (its full budget); PTO arms and GRPO K=0 run to 10.
`train_iter n` = branching by policy π_n = the policy the eval calls `model_iter_{n−1}`; train_iter 1 is the base
policy in every arm, so it is the one same-policy row where the two K arms differ only in the reward measurement.

In [ ]:
import sys, os
_p = os.path.abspath(".")                      # find eda/ (the dir holding eda_analysis/) from any depth
while _p != os.path.dirname(_p) and not os.path.isdir(os.path.join(_p, "eda_analysis")):
    _p = os.path.dirname(_p)
sys.path.insert(0, _p)
import warnings; warnings.filterwarnings("ignore")
import time
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
pd.set_option("display.width", 185, "display.max_columns", 50)

import os, eda_analysis
from eda_analysis import exports, plotting
cfg = eda_analysis.EdaConfig(family="lookahead/mechanism", judge=os.environ.get("EDA_JUDGE", ""))
S = eda_analysis.notebook_setup(cfg)
exports.reset_results()   # clears only THIS family's generated figures/tables (never SUMMARY.md)
exports.save_provenance(cfg, S.SCORES)   # reset_results wipes figures/, incl. the banner notebook_setup just wrote — re-stamp it

from eda_analysis import pref, behavior, training, dispersion, faithfulness, tails, reliability as R
from eda_analysis.constants import BOOT_SEED
_T0 = time.time()

# The four canonical arms (both K arms of both optimizers). Everything here is keyed by them; extra arms on
# disk (if any) are ignored so the fixture comparisons stay well-defined.
CANON = list(dispersion.ARMS)                                        # PTO_LA0 PTO_LA5 GRPO_LA0 GRPO_LA5
KA = [a for a in eda_analysis.cross_k_arms(S) if a.label in CANON]   # both K arms of every method
KA_LABELS = [a.label for a in KA]
missing = [a for a in CANON if a not in KA_LABELS]
if missing:
    print(f"[mechanism] NOTE: arms missing on disk: {missing} — the K contrasts that need them are skipped.")
print("cross-K arms:", KA_LABELS)

# Both graders, same arm/metric filters, primary FIRST — the EVAL side of §3's join.
SC = eda_analysis.scores_by_judge(S)
PRIMARY_LABEL = eda_analysis.constants.judge_dirname("")            # 'gpt-4o-mini'
HELDOUT_TAGS = R.second_judge_tags()
HELDOUT_TAG = HELDOUT_TAGS[0] if HELDOUT_TAGS else None             # 'anthropic_claude-haiku-4-5'
HELDOUT_LABEL = eda_analysis.constants.judge_dirname(HELDOUT_TAG) if HELDOUT_TAG else None
HELDOUT_NAME = R.judge_display(HELDOUT_TAG) if HELDOUT_TAG else None
if len(HELDOUT_TAGS) > 1:
    print(f"[mechanism] NOTE: {len(HELDOUT_TAGS)} held-out judges on disk; §3 puts every one of them beside the primary "
          f"(figures + wide tables for the first, {HELDOUT_TAG}).")
SC = {k: v[v["arm"].isin(KA_LABELS)] for k, v in SC.items()}
print(f"primary = {PRIMARY_LABEL} {SC[PRIMARY_LABEL].shape}"
      + (f" | held-out = {HELDOUT_LABEL} ({HELDOUT_NAME}) {SC[HELDOUT_LABEL].shape}" if HELDOUT_LABEL else " | no held-out judge on disk"))

# The promoted paper figures (dispersion, dispersion_tau, faithfulness*, tail_audit) are laid out for a 7.2-inch,
# default-font page (the paper's rc); under the EDA-wide seaborn 'notebook' context their axis labels crowd
# the neighbouring panels / overflow the tight bbox, so they render under the 'paper' context (fonts ~0.8x) —
# same data, same statistics, the paper's look (precedent: lookahead/replication sd_fig).
PAPER_RC = sns.plotting_context("paper")
TRAIN_GRADER = dispersion.GRADER                                    # 'training oracle (gpt-4o-mini)'
GROUP_UNIT = ("Pairing unit = the GROUP: the M=8 (PTO) / G=8 (GRPO) candidates the policy sampled at one branch point, keyed "
              "(arm, train_iter, conversation_id, branch_id, epoch) — PTO's branch_id is the trunk depth, so conversation_id "
              "is required.")
CENSOR = "GRPO_LA5 is right-censored at train_iter 5 (its full budget); PTO arms and GRPO_LA0 run to 10."
ITER_AXIS = ("train_iter n = branching by policy pi_n, the policy the eval calls model_iter_{n-1}; train_iter 1 = the BASE "
             "policy in every arm (the one same-policy row).")

## 1 · The over-praise chain: select → generate → evaluate, K=0 vs K=5  `[TRAINING] → [EVAL]`

**Purpose.** `arms/preference` shows the reward-hacking mechanism as a two-level gap (what the update selects
for vs what the policy generates) per arm. This runs the *whole* chain for both look-ahead arms of PTO, on the
one channel the hack travels through (over-praise), because a claim that a knob **prevents** a behaviour is only
mechanistic if the knob is visible at the **first** level. A behaviour has to clear three gates to become policy:

1. **what the REWARD selects for** — `w_overpraise` from `pref.weighted_lexical_contrast`: Σ w·feature per
   gradient group (DPO's ±1 chosen/rejected roles, rescaled to Σ|w| = 2), ± SE over groups. If K=5's selection
   weight is flat where K=0's rises, look-ahead changed *the reward*, and everything downstream follows. If both
   weights rise and only the outcome differs, the story is something else and this table will say so.
2. **what the POLICY generates** — `pool_overpraise` from `pref.pool_mean_by_iter`: the unweighted mean over
   *every* candidate (incl. the τ-dropped ones), i.e. what the policy produces before any selection.
3. **what the EVAL measures** — the per-therapist-turn over-praise rate on the full conversations
   (`behavior.mici_behavior_by_iter`, `MICI_OverPraise_rate`), the thing `lookahead/behaviour` tests.

**Indexing.** Rows 1–2 are indexed by `train_iter n`, which samples from the **iter-start** policy — the same
policy the eval calls `model_iter_{n−1}`. Table and figure shift them by −1 so all three rows share one policy
axis; without that the levels are off by one and the chain appears to run backwards.

**Grader.** Rows 1–2 are judge-invariant by construction (`generations.jsonl` records what the *training* oracle
selected during the run). Row 3 is the primary oracle here; its held-out-judge counterpart is the over-praise
trajectory in `lookahead/behaviour`. Exact, embedding-free — the semantic-direction probes stay in
`arms/preference`.

> ⚠️ **A mechanism, not a causal test.** Nothing here randomises anything except K itself, which is the
> intervention. The chain being consistent at all three levels is strong evidence that look-ahead acts on the
> reward rather than on the policy directly — it is not proof that no other pathway contributes.

In [ ]:
KC_ALL = pref.load_weighted_candidates(KA, drop_zero_weight=False)   # pool: every candidate (incl. w = 0)
KC     = pref.load_weighted_candidates(KA)                           # push: gradient groups only
KPOOL  = pref.pool_mean_by_iter(KC_ALL)
KPUSH  = pref.weighted_lexical_contrast(KC)
KEVAL  = behavior.mici_behavior_by_iter(KA)                          # primary oracle, per therapist turn
print(f"candidates: {len(KC_ALL):,} (all) / {len(KC):,} (weighted); arms with training generations: "
      f"{sorted(KPOOL.arm.unique()) if not KPOOL.empty else 'none'}   [{time.time() - _T0:.0f} s]")

PANEL_ARMS = [a for a in ["PTO_LA0", "PTO_LA5"] if a in set(KPOOL.arm.unique())]
if len(PANEL_ARMS) < 2:
    print("need both PTO_LA0 and PTO_LA5 with training generations for the chain "
          f"(have: {sorted(KPOOL.arm.unique())}).")
else:
    # The three levels side by side as a table too — the figure is the readable form, but a number quoted in a
    # write-up should come from an artifact, not off a plot.
    CHAIN = (KPUSH[["arm", "train_iter", "w_overpraise", "w_overpraise_se"]]
             .merge(KPOOL[["arm", "train_iter", "pool_overpraise", "pool_overpraise_se"]],
                    on=["arm", "train_iter"], how="outer")
             .assign(iteration=lambda d: d.train_iter - 1)          # onto the POLICY axis
             .merge(KEVAL[["arm", "iteration", "MICI_OverPraise_rate"]],
                    on=["arm", "iteration"], how="outer")
             .sort_values(["arm", "iteration"]))
    CHAIN = CHAIN[CHAIN.arm.isin(PANEL_ARMS)].reset_index(drop=True)
    display(CHAIN.round(4))
    exports.save_table(CHAIN.round(4), "k_mechanism_overpraise_chain", float_format="%.4f", caption=(
        "**The over-praise channel at all three levels it must pass to become policy**, per PTO look-ahead arm: "
        "`w_overpraise` = what the training reward SELECTS for (Sum(w*feature) per gradient group, DPO's +/-1 "
        "chosen/rejected roles rescaled to Sum|w| = 2, +/- SE over groups; `pref.weighted_lexical_contrast`), "
        "`pool_overpraise` = what the policy GENERATES (unweighted mean over EVERY candidate incl. the tau-dropped "
        "ones, +/- SE; `pref.pool_mean_by_iter`), and `MICI_OverPraise_rate` = what the full-conversation EVAL "
        "measures per therapist turn (`behavior.mici_behavior_by_iter`). The two training columns are indexed by "
        "train_iter, which samples the ITER-START policy, so they are shifted onto the policy axis as iteration = "
        "train_iter - 1 and every row describes ONE policy (the final model state has an eval row and no training "
        f"row). {GROUP_UNIT} Training columns are judge-invariant by construction (generations.jsonl records the "
        f"{TRAIN_GRADER}'s own selection); the eval column is the primary oracle (gpt-4o-mini) — its held-out-judge "
        "counterpart is the over-praise trajectory in lookahead/behaviour. Reads: K=5's selection weight staying "
        "flat where K=0's rises = look-ahead changes the REWARD and the pool and the eval follow. Ported from the "
        "retired 6_Preference section 5d (results/L5/tables/6_preference/gpt-4o-mini/k_mechanism_overpraise_chain)."))

    fig = plotting.k_mechanism_panel(
        KPUSH, KPOOL, KEVAL,
        select_col="w_overpraise", generate_col="pool_overpraise",
        evaluate_col="MICI_OverPraise_rate", arms=PANEL_ARMS,
        suptitle="Over-praise: where look-ahead intervenes (PTO, K=0 solid · K=5 dashed)")
    exports.save_fig(fig, "k_mechanism_overpraise", caption=(
        "**The reward-hack channel at all three levels, both PTO look-ahead arms.** Top: what the training reward "
        "selects for within a group (+/-1 SE over groups). Middle: what the policy generates over every candidate. "
        "Bottom: what the full-conversation eval measures per therapist turn (primary oracle). Training rows are "
        "shifted -1 onto the policy axis so all three describe the same policy; K=0 solid, K=5 dashed. K=5's "
        "selection weight staying flat where K=0's rises is the mechanistic claim: look-ahead changes the REWARD, "
        "and the pool and the eval follow. The top two rows are judge-invariant (they record the training oracle's "
        f"own choices). {GROUP_UNIT} Table: `k_mechanism_overpraise_chain`."))
    plt.show()

## 2 · Dispersion — does look-ahead WIDEN the training signal, or merely RESCALE it?  `[TRAINING]`

**Purpose.** Guards against the "look-ahead sharpens the signal" misreading. If look-ahead only multiplied the
within-group score spread by a constant, then (a) the best–worst margin and the within-group SD rise by the SAME
factor, (b) margin/SD stays at the value expected for M=8 iid draws (pure sampling spread), and (c) part of K=5's
higher PTO pair yield is an artefact of the ABSOLUTE τ = 0.1 filter applied to margins that were rescaled up. All
three are tested here, on every candidate the trainers scored (training phase only — GRPO's TRL `eval`-loop
generations are dropped, mirroring `pref.pair_yield_by_iter`).

**Estimators.** Within-group SD is the POPULATION SD (ddof=0) — exactly what GRPO records as `group_std` (the
notebook re-asserts it) and what `scale_rewards="group"` divides by; the same estimator is applied to PTO. Margin
= max − min of the group's scores (τ-free, over ALL 8 candidates). The iid-normal expectation for margin/SD and
for the winner's standardized lead `(max − mean)/SD` is SIMULATED (200k groups of 8) with the same estimator; a
distribution-free shuffle null (scores permuted across groups within an arm-iteration) is reported alongside
because the oracle score is bounded and discrete.

**Direction.** Ratios are **K5 / K0** (>1 = K=5 wider); `winner_z_diff` = K5 − K0 (+ = K=5's winner stands out
more). This deliberately differs from the rubric-contrast convention (+ ⇒ K=0 higher) used in `lookahead/reward`.
Higher SD / margin = wider spread, NOT better discrimination. Only `train_iter 1` compares the same policy under
the two K measurements; later rows compare diverged policies.

In [ ]:
GENS = training.load_generations(KA, keep_tail=False)     # memoized; §1 already read the same rows
G    = dispersion.load_group_frame(KA, gens=GENS)          # one row per group
print(f"{len(GENS):,} candidates -> {len(G):,} groups   [{time.time() - _T0:.0f} s]")

EXP = dispersion.iid_expectation()                                            # pure geometry of 8 draws
E0 = EXP[EXP.sd_estimator == "ddof=0"].iloc[0]
EXP_ROM, EXP_WZ = float(E0.ratio_of_means), float(E0.winner_z_mean)
exports.save_table(EXP, "dispersion_expectation", float_format="%.4f", caption=(
    "**Simulated reference for the best-worst margin over within-group SD when the M=8 candidate scores are iid "
    "normal** (pure sampling spread, no true separation between candidates): 200,000 simulated groups; "
    "`ratio_of_means` = E[range]/E[SD] (the estimator used for `margin_over_sd` in `dispersion_by_iter`), "
    "`mean_of_ratios` = E[range/SD]. `ddof=0` is the population SD GRPO records as `group_std` and divides its "
    "advantages by; `ddof=1` is the sample SD; `winner_z_mean` = E[(max - mean)/SD], the standardized lead of the "
    "best candidate over its group. Neither depends on the grader (pure geometry of 8 draws). Simulation seed = "
    "constants.BOOT_SEED (the paper fixture used 123; values agree to ~1e-3)."))

BYIT = dispersion.dispersion_by_iter(KA, gens=GENS)
print(f"[check] GRPO recorded group_std vs ddof=0 SD: max |diff| = {BYIT.attrs['grpo_group_std_max_abs_diff']:.2e}")
_m5 = BYIT[(BYIT.arm == "PTO_LA5") & (BYIT.train_iter == 1)]
_m0 = BYIT[(BYIT.arm == "PTO_LA0") & (BYIT.train_iter == 1)]
if len(_m5) and len(_m0):
    print(f"[check] mean best-worst margin at train_iter 1: PTO_LA5 {_m5.margin_mean.iloc[0]:.4f} (fixture 0.4242), "
          f"PTO_LA0 {_m0.margin_mean.iloc[0]:.4f} (fixture 0.2735)")
    assert abs(_m5.margin_mean.iloc[0] - 0.4242) < 5e-4 and abs(_m0.margin_mean.iloc[0] - 0.2735) < 5e-4, "fixture anchor moved"
display(BYIT[["arm", "train_iter", "n_groups", "sd_mean", "margin_mean", "margin_over_sd", "winner_z", "yield_tau0.10"]].round(3))
exports.save_table(BYIT, "dispersion_by_iter", caption=(
    f"**Within-group dispersion of the TRAINING reward** (Q1+Q2 mean, 0-5 oracle scale, {TRAIN_GRADER}) per arm x "
    f"training iteration. {GROUP_UNIT} TRL eval-phase generations excluded. `sd_*` = population SD (ddof=0, GRPO's "
    "logged `group_std`) of the group's scores; `margin_*` = best - worst score in the group over ALL candidates "
    f"(tau-free); `margin_over_sd` = mean margin / mean SD (compare with the iid-normal expectation {EXP_ROM:.3f} for "
    f"8 draws); `mean_group_ratio` = mean of per-group margin/SD over groups with SD>0; `winner_z` = mean of (best - "
    f"group mean)/SD over groups with SD>0 (iid-normal expectation {EXP_WZ:.3f}; the 'does the winner stand out?' "
    "statistic); `null_margin_over_sd` / `null_winner_z` = the same two statistics after shuffling that "
    "arm-iteration's candidate scores across groups (100 permutations, seed = BOOT_SEED; a distribution-free "
    "no-within-group-structure reference that keeps the arm's actual bounded, discrete score distribution — NB "
    "`null_winner_z` is sensitive to the skew of the pooled score distribution, so compare `winner_z` across K arms "
    "on the raw value); `frac_sd_zero` = share of groups whose 8 scores are identical; `frac_groups_floored` = share "
    "of groups with a candidate at REWARD_FLOOR; `frac_groups_nan_cand` = share of groups with an unscored candidate "
    "(dropped within the group; `n_groups` counts groups with >= 2 valid scores, `n_groups_all` every group logged); "
    "`yield_tau0.10` = share of groups with margin > 0.10 (PTO's tau; informative-only for GRPO); `reward_*` = "
    f"per-candidate score median / IQR. {ITER_AXIS} `eval_iter` = train_iter - 1. {CENSOR} Higher SD/margin = wider "
    "spread, NOT better discrimination. Fixture: papers/2026_lookahead_pto_grpo/tables/dispersion_by_k_by_iter.csv "
    "(point estimates exact; the null columns are re-drawn under BOOT_SEED)."))

RAT = dispersion.dispersion_ratios(G)
display(RAT[["method", "train_iter", "n_groups_K0", "n_groups_K5", "margin_ratio", "sd_ratio", "ratio_of_ratios",
             "ror_lo", "ror_hi", "winner_z_diff"]].round(3))
exports.save_table(RAT, "dispersion_ratios", caption=(
    "**The 'same factor' test**: K=5 / K=0 ratio of the mean best-worst margin and of the mean within-group SD "
    f"(ddof=0), per method x training iteration, {TRAIN_GRADER}. Ratio > 1 = K=5 wider. `ratio_of_ratios` = "
    "margin_ratio / sd_ratio (1.0 = look-ahead scales margin and SD by the same factor, i.e. it rescales the spread "
    "without pulling the winner away from the pack); 95% CIs are percentile bootstraps over groups (2,000 resamples, "
    "seed = constants.BOOT_SEED, groups resampled independently within each arm, the same resample driving margin and "
    "SD). `pooled` = all iterations both arms have (GRPO: 1-5, right-censored; PTO: 1-10); train_iter 1 is the "
    "same-policy (base) row and the only rescaling factor free of policy divergence. `margin_over_sd_K*` compares to "
    f"the iid-normal expectation {EXP_ROM:.3f}; `winner_z_*` = mean (best - group mean)/SD (groups with SD>0; "
    f"iid-normal expectation {EXP_WZ:.3f}) and its K5 - K0 difference with bootstrap CI (+ = K=5's winner stands out "
    f"more, in within-group SD units) — the direct 'sharper signal' statistic. {GROUP_UNIT} {CENSOR} Fixture: "
    "papers/2026_lookahead_pto_grpo/tables/dispersion_by_k_ratios.csv (ratios exact; CI bounds re-drawn)."))

TAU = dispersion.tau_sensitivity(G, ratios=RAT)
R_ITER1 = float(TAU.attrs["r_iter1"])
print(f"rescale factor r_iter1 (K5/K0 within-group SD ratio at train_iter 1, PTO) = {R_ITER1:.4f}")
display(TAU[TAU.tau == dispersion.TAU_TRAINER][["train_iter", "tau", "yield_K0_raw", "yield_K5_raw", "gap_K5_minus_K0",
                                                 "yield_K0_x_r1", "share_gap_closed_r1"]].round(3))
exports.save_table(TAU, "dispersion_tau", caption=(
    "**PTO tau-sensitivity**: pair yield = share of branch points (groups) whose best-worst margin exceeds tau "
    f"(strict >, as in the trainer; the run used tau = {dispersion.TAU_TRAINER}), {TRAIN_GRADER}, per training "
    "iteration and pooled over iterations 1-10. `yield_K*_raw` on the recorded margins; `yield_K0_x_r1` after "
    f"multiplying every K=0 margin by r1 = {R_ITER1:.3f} (the K=5/K=0 within-group SD ratio at train_iter 1, where "
    "both arms are the base policy); `yield_K5_div_r1` after dividing K=5 margins by the same factor; "
    "`yield_K0_x_r_iter` uses each iteration's own SD ratio. `share_gap_closed_*` = (rescaled K=0 yield - raw K=0 "
    "yield) / (raw K=5 yield - raw K=0 yield): the fraction of K=5's yield advantage reproduced by pure rescaling of "
    "K=0's spread (1.0 = all of it; can exceed 1 when rescaling overshoots; NaN when the raw gap is ~0 — iterations "
    "8-10 have a raw gap ~0 or negative because K=5's own spread shrank as its policy diverged, so quote the gap in "
    f"yield points there). Groups with < 2 valid candidate scores excluded. {GROUP_UNIT} Not a comparison of "
    "policies at iterations >= 2 (the arms have diverged) — only train_iter 1 is same-policy. Fixture: "
    "papers/2026_lookahead_pto_grpo/tables/dispersion_by_k_tau.csv (exact)."))

with PAPER_RC:
    fig = plotting.dispersion_fig(BYIT, EXP, arms=[a for a in CANON if a in set(BYIT.arm)], grader=TRAIN_GRADER)
exports.save_fig(fig, "dispersion", caption=(
    "**Within-group dispersion of the training reward by training iteration, four arms (2x2).** (a) within-group SD "
    "(ddof=0) of the 8 candidate scores; (b) best-worst margin; (c) margin over SD with the iid-normal expectation "
    f"({EXP_ROM:.2f}) dotted and the shuffle-null range shaded; (d) the winner's standardized lead (best - group "
    f"mean)/SD with its iid-normal expectation ({EXP_WZ:.2f}) dotted. K=0 solid + circle, K=5 dashed + square, colour "
    f"= arm. {TRAIN_GRADER}. Read: K=5 raises SD and margin together (a, b) while (c) and (d) stay near the iid "
    f"reference — the spread is rescaled, the winner does not stand out more. {GROUP_UNIT} {CENSOR} Tables: "
    "`dispersion_by_iter`, `dispersion_expectation`."))
plt.show()

with PAPER_RC:
    fig = plotting.tau_fig(TAU, grader=TRAIN_GRADER)
exports.save_fig(fig, "dispersion_tau", caption=(
    "**PTO pair yield vs tau, raw and rescaled.** (a) train_iter 1 (base policy, same in both arms); (b) all "
    "iterations 1-10 pooled. K=0 solid + circle, K=5 dashed + square (raw margins); open markers = K=0 margins x "
    f"{R_ITER1:.2f} (dotted) and K=5 margins / {R_ITER1:.2f} (dash-dot), the iteration-1 within-group SD ratio; "
    f"vertical dotted line = the tau used in training ({dispersion.TAU_TRAINER}). {TRAIN_GRADER}. Read: the share of "
    "K=5's yield advantage the rescaled K=0 curve reproduces is the part of the 'more pairs under look-ahead' story "
    f"that is a scale artefact of the absolute tau filter. {GROUP_UNIT} Table: `dispersion_tau`."))
plt.show()

## 3 · Reward faithfulness — is the partial-conversation proxy faithful to the full-conversation eval, and does K help?  `[TRAINING] → [EVAL]`

**Purpose.** Both trainers score *partial* conversations (a prefix + one completion, + 5 simulated turns under
K=5) as the training reward, but the thesis evaluates *full* conversations. `arms/training` shows the per-arm
reliability curve; this section turns it into TABLES with cluster-bootstrap CIs, a stated unit, a matched-policy
cut, a persona-cooperation cut and the proxy-vs-eval level table — under BOTH graders side by side — and asks
whether look-ahead makes the proxy *more faithful* (the mechanism the ICLR paper posits) or merely different.

**Unit.** A *branch row* = one branch point of the training run: a prefix of `n_turns` utterances (therapist +
patient, ending on a patient turn) plus the 8 completions sampled from the iter-start policy. `proxy_score` = the
training oracle's score of the CHOSEN (arg-max) completion on `prefix + completion` (K=0) or `prefix + completion
+ 5 simulated turns` (K=5) — read from `generations.jsonl`, no new oracle calls. `eval_score` = full-conversation
Q1Q2 of the eval conversation the prefix was cut from (`model_iter_{train_iter−1}`, joined on
`conversation_id == file_index`, same persona). `agreement` = fraction of conversation PAIRS within one
`(arm, eval_iter, n_turns)` cell whose proxy ordering matches their eval ordering (ties dropped), counts pooled
over eval_iters; 0.5 = chance. CIs = 95 % cluster bootstrap over conversations within each model state (B=1000,
seed = `BOOT_SEED`; pairs within a cell are NOT independent, so `n_pairs` never feeds a CI). Point estimates
reproduce `stats.rank_agreement_by_nturns` bin-for-bin (asserted below).

**Graders.** The proxy is ALWAYS the training oracle's score (it cannot be re-graded). The eval side is computed
under every grader in the score lake — primary = gpt-4o-mini (same-grader faithfulness), held-out = Claude Haiku
4.5 (cross-grader faithfulness). Never averaged.

**Sign.** delta = K0 − K5 (**+ ⇒ K=0 higher**); for faithfulness a NEGATIVE delta means look-ahead helped.
Only `train_iter 1` (cut `train_iter_1`) compares the SAME base policy under the two K measurements; the
`iters_1-5` / `matched_iters` cuts pool diverged policies. GRPO_LA5 is right-censored at iteration 5, so GRPO_LA0
is shown both on its full support (iters 1–10) and restricted to 1–5 (like-for-like).

**Caveats.** PTO greedy trunks share only the first MCL=12 utterances with the eval conversation (beyond that the
prefix follows the best-of-M trunk, so PTO's curve mixes cut length with trunk divergence; GRPO prefixes are
exact slices). GRPO branch rows include the policy drifting within an iteration (2 epochs) and the ~3–10 %
eval-split groups TRL scores at iteration end. GRPO_LA5 iteration 1 captured only its second epoch (crash +
resume). Wilcoxon over `n_turns` bins treats correlated bins as observations — descriptive only.

In [ ]:
FD = faithfulness.faithfulness_data(KA, SC)                          # ~2.5 min: re-reads every generations.jsonl prefix
print(f"branch rows {len(FD.BR):,}; joined per grader: "
      + ", ".join(f"{j} {len(FD.DF[j]):,}" for j in FD.judges) + f"   [{time.time() - _T0:.0f} s]")
MAX_DEV = faithfulness.check_against_rank_agreement(FD, SC[PRIMARY_LABEL])
print(f"[selfcheck] point estimates reproduce stats.rank_agreement_by_nturns bin-for-bin (max |dev| = {MAX_DEV:.2e})")

JUDGES = list(FD.judges)                                             # primary first
GRADERS = ("Proxy = the training oracle (gpt-4o-mini) by construction; eval side under the grader named in the table "
           f"(primary = {PRIMARY_LABEL}" + (f", held-out = {HELDOUT_NAME}" if HELDOUT_NAME else "") + "). Never averaged across graders.")
UNIT = faithfulness.UNIT_NOTE
SIGN = faithfulness.SIGN_NOTE
CENSOR_F = faithfulness.CENSOR_NOTE
CUT = faithfulness.CUT_NOTE

# ── 3a · the curve: agreement vs prefix length, pooled over iterations ────────────────────────
CURVE = faithfulness.faithfulness_curve(FD)
_c = CURVE[(CURVE.judge == PRIMARY_LABEL) & (CURVE.arm == "PTO_LA0") & (CURVE.iters == "1-10") & (CURVE.n_turns == "12")]
if len(_c):
    print(f"[check] PTO_LA0 n_turns=12 agreement (primary) = {_c.agreement.iloc[0]:.4f}, n_pairs {int(_c.n_pairs.iloc[0])} "
          "(fixture 0.8647, n=33323)")
    assert abs(_c.agreement.iloc[0] - 0.8647) < 5e-4 and int(_c.n_pairs.iloc[0]) == 33323, "fixture anchor moved"
exports.save_table(CURVE, "faithfulness_curve_long", caption=(
    "**Long form of the reward-faithfulness curve**: sign-agreement between the training proxy and the "
    "full-conversation eval Q1Q2, per (grader of the eval side, arm, iters range, n_turns bin), pooled over "
    "training iterations; rows n_turns='all' pool every bin, '12-20'/'22-34'/'36-50' pool coarse ranges; bins with "
    f"< 20 pairs dropped. {UNIT} {GRADERS} {CENSOR_F} GRPO_LA0 appears on its full support (iters 1-10) AND "
    "restricted to iters 1-5 (like-for-like with censored GRPO_LA5). Fixture: "
    "papers/2026_lookahead_pto_grpo/tables/reward_faithfulness_curve_long.csv (agreement + n_pairs exact; CIs "
    "re-drawn under BOOT_SEED)."))
for j in JUDGES:
    name = "faithfulness_curve" if j == PRIMARY_LABEL else ("faithfulness_curve_heldout" if len(JUDGES) == 2 else f"faithfulness_curve_{j}")
    W = faithfulness.curve_wide(CURVE, j)
    if j == PRIMARY_LABEL:
        display(W)
    exports.save_table(W, name, caption=(
        ("**Reward faithfulness vs partial-conversation length — same-grader** (eval side = the training oracle, "
         "gpt-4o-mini). " if j == PRIMARY_LABEL else
         f"**Reward faithfulness vs partial-conversation length — CROSS-grader** (eval side = the held-out judge, "
         f"{faithfulness.judge_display(j)}; the proxy is still the training oracle's score). ")
        + "Cells: agreement [95% CI]; `... pairs` = number of conversation pairs pooled over iterations (the column "
        f"header names the train_iter range pooled). {UNIT} {CENSOR_F} GRPO_LA0 is shown both on its full support "
        "(iters 1-10) and restricted to iters 1-5 (like-for-like with GRPO_LA5). Bins with n_pairs < 20 are not "
        "reported. Long form: `faithfulness_curve_long`."))

# ── 3b · per training iteration + the K contrast per iteration ────────────────────────────────
BYITER = faithfulness.faithfulness_by_iter(FD)
exports.save_table(BYITER, "faithfulness_curve_by_iter_long", float_format="%.4f", caption=(
    "**Reward faithfulness per (arm, train_iter), long form**: all n_turns bins pooled + the three coarse ranges "
    "(primary-grader columns `agreement, ci_lo, ci_hi, n_pairs, n_convs, n_bins, agr_/lo_/hi_/pairs_<range>`; each "
    "held-out grader adds `agreement<sfx>, ci_lo<sfx>, ci_hi<sfx>, n_pairs<sfx>`, sfx = `_heldout` for a single "
    f"held-out grader). {ITER_AXIS} {UNIT} {GRADERS} {CENSOR_F} Presentation: `faithfulness_curve_by_iter`."))
BYITER_D = faithfulness.by_iter_display(BYITER)
display(BYITER_D)
exports.save_table(BYITER_D, "faithfulness_curve_by_iter", caption=(
    "**Reward faithfulness per training iteration** (all n_turns bins pooled, plus three coarse ranges): agreement "
    "[95% CI] with the eval side under the training oracle (gpt-4o-mini) and, last column(s), under the held-out "
    "judge; coarse-range cells with < 20 pairs are blank. train_iter n branches from policy pi_{n-1}, whose eval "
    "conversations are model_iter_{n-1} = eval_iter; train_iter 1 = the BASE policy for every arm (a matched-policy "
    f"row). {UNIT} {CENSOR_F} Fixture: papers/2026_lookahead_pto_grpo/tables/reward_faithfulness_curve_by_iter.csv."))

DK = faithfulness.k_faithfulness_by_iter(FD)
DK_D = faithfulness.k_by_iter_display(DK)
display(DK_D)
exports.save_table(DK_D, "faithfulness_k_by_iter", caption=(
    "**K=0 vs K=5 faithfulness per training iteration** (all bins pooled), per method and eval-side grader. "
    f"{SIGN} CI = percentile of the difference of independent cluster-bootstrap replicates (the two arms are "
    "different conversation draws; seed = BOOT_SEED). Only train_iter 1 samples the SAME policy in both K arms; "
    f"later rows compare diverged policies. {UNIT} {GRADERS} {CENSOR_F} Fixture: "
    "papers/2026_lookahead_pto_grpo/tables/reward_faithfulness_k_by_iter.csv (agr_K0/agr_K5/delta exact)."))

# ── 3c · matched policy per bin + the summary ─────────────────────────────────────────────────
MP, TESTS = faithfulness.matched_policy(FD)
exports.save_table(MP, "faithfulness_matched_policy_long", float_format="%.4f", caption=(
    "**K=0 vs K=5 reward faithfulness per n_turns bin under three iteration cuts, long form** (`agr_K0, K0_lo, "
    f"K0_hi, pairs_K0, agr_K5, K5_lo, K5_hi, pairs_K5, delta_K0_minus_K5, d_lo, d_hi`). {CUT} {SIGN} {UNIT} "
    f"{GRADERS} {CENSOR_F} Presentation: `faithfulness_matched_policy`."))
MP_D = faithfulness.matched_policy_display(MP)
display(MP_D[MP_D.cut == "train_iter_1"])
exports.save_table(MP_D, "faithfulness_matched_policy", caption=(
    f"**K=0 vs K=5 reward faithfulness per n_turns bin under three iteration cuts.** {CUT} {SIGN} Per-bin CI = "
    "percentile of the difference of independent cluster-bootstrap replicates (the two arms are different "
    f"conversation draws). Bins with < 20 pairs in either arm are dropped. {UNIT} {GRADERS} {CENSOR_F} Fixture: "
    "papers/2026_lookahead_pto_grpo/tables/reward_faithfulness_matched_policy.csv (point estimates exact; CIs "
    "re-drawn — 4 of ~1,700 cells move by > 0.02 in thin train_iter_1 bins)."))
display(TESTS.round(4))
exports.save_table(TESTS, "faithfulness_matched_policy_tests", float_format="%.4f", caption=(
    "**Wilcoxon signed-rank test over n_turns BINS** (paired by bin; the per-bin deltas of "
    f"`faithfulness_matched_policy`) of delta = agreement(K0) - agreement(K5). {SIGN} {CUT} Bins are NOT "
    "independent observations (the same conversations feed neighbouring bins), so read p as descriptive; the "
    "per-bin CIs and `faithfulness_k_summary` carry the inference. n_bins = bins with >= 20 pairs in both arms. "
    f"{GRADERS} {CENSOR_F} Fixture: papers/2026_lookahead_pto_grpo/tables/reward_faithfulness_matched_policy_tests.csv (exact)."))

KSUM = faithfulness.k_summary(FD)
exports.save_table(KSUM, "faithfulness_k_summary_long", float_format="%.4f", caption=(
    "**Summary of the K contrast in reward faithfulness, long form** (all n_turns bins pooled) per eval-side "
    f"grader, method and iteration cut. {CUT} {SIGN} {UNIT} {GRADERS} {CENSOR_F} Presentation: `faithfulness_k_summary`."))
KSUM_D = faithfulness.k_summary_display(KSUM)
display(KSUM_D)
exports.save_table(KSUM_D, "faithfulness_k_summary", caption=(
    "**SUMMARY of the K contrast in reward faithfulness** (all n_turns bins pooled) per eval-side grader, method and "
    f"iteration cut. {CUT} 'delta pooled pairs' weights every conversation pair equally (arms with more branch "
    "points at some iterations weigh those iterations more); 'delta iter-mean' is the mean of the per-iteration "
    "K0-K5 deltas (equal weight per training iteration; CI from the same replicates), with a Wilcoxon over "
    f"iterations (n_iters >= 5 only) and the count of iterations on each side. {SIGN} {UNIT} {GRADERS} {CENSOR_F} "
    "Fixture: papers/2026_lookahead_pto_grpo/tables/reward_faithfulness_k_summary.csv (deltas / counts / p exact; "
    "CIs re-drawn under BOOT_SEED)."))

# ── 3d · by patient cooperation level (primary grader) ────────────────────────────────────────
COOP = faithfulness.by_cooperation(FD)
exports.save_table(COOP, "faithfulness_by_coop_long", float_format="%.4f", caption=(
    "**Reward faithfulness by patient cooperation level, long form** (`n_convs_mean, agreement, ci_lo, ci_hi, n_pairs`, "
    "coarse-range `agr_/lo_/hi_/pairs_<range>`, matched-policy `agr_iter1/lo_iter1/hi_iter1/pairs_iter1`); the "
    "within-stratum K deltas are in `faithfulness_numbers.json` (by_coop.*). Eval side = the training oracle "
    f"(gpt-4o-mini). {UNIT} {CENSOR_F} Presentation: `faithfulness_by_coop`."))
COOP_D = faithfulness.by_cooperation_display(COOP)
display(COOP_D)
exports.save_table(COOP_D, "faithfulness_by_coop", caption=(
    "**Reward faithfulness by patient cooperation level** (32 personas each: Cooperative = 'High', Warms up = "
    "'StartLowAndChangesToHigh', Resistant = 'Low'; persona attached to each branch row via the eval conversation's "
    "file_index -> persona_id). Pairs are formed WITHIN a cooperation stratum, so the statistic asks whether the "
    "training proxy ranks same-cooperation conversations like the full-conversation eval does (the easy "
    "between-strata ordering is removed). Eval side = the training oracle (gpt-4o-mini). agreement [CI] pools all "
    "n_turns and iterations; the coarse-range columns split by cut length; the last two columns are the "
    "matched-policy (train_iter 1, base policy) cut. n_convs_mean = mean conversations per (eval_iter, n_turns) "
    f"cell. Each stratum's bootstrap is seeded BOOT_SEED + 1 + stratum index. {UNIT} {CENSOR_F} Fixture: "
    "papers/2026_lookahead_pto_grpo/tables/reward_faithfulness_by_coop.csv (agreement / pairs exact)."))

# ── 3e · proxy vs eval LEVELS ─────────────────────────────────────────────────────────────────
LV, RHO = faithfulness.proxy_levels(FD)
display(LV.round(3))
exports.save_table(LV, "faithfulness_levels", caption=(
    "**Proxy-vs-eval LEVELS per arm x model state**: `proxy_mean` = mean over that iteration's branch points of the "
    "CHOSEN (arg-max) candidate's training-oracle score (K=0: prefix+completion; K=5: the K-EXTENDED score, "
    "prefix+completion+5 simulated turns) — the reward the update actually optimised, indexed by the policy that "
    "produced it (train_iter = eval_iter + 1; the final model state, eval_iter 10 / GRPO_LA5 5, was evaluated but "
    "never trained on, hence NaN proxy). `eval_mean` (primary, gpt-4o-mini) / `eval_mean_heldout` (Claude Haiku "
    "4.5) = mean full-conversation Q1Q2 of the same model state under each grader — never averaged. `gap_*` = "
    "proxy_mean - eval_mean (+ => the training reward reads higher than the full-conversation eval). "
    "`mean_n_turns` = mean prefix length of the branch points. Iteration 0 = two independent base draws per method "
    f"(one per K arm). {CENSOR_F} Fixture: papers/2026_lookahead_pto_grpo/tables/reward_faithfulness_levels.csv (exact)."))
display(RHO.round(3))
exports.save_table(RHO, "faithfulness_levels_rho", caption=(
    "**Across-iteration association between the training-proxy LEVEL and the full-conversation eval LEVEL** (rows of "
    "`faithfulness_levels` with a proxy), per arm and eval-side grader: Spearman rho and Pearson r over n_iters model "
    "states (train_iter 1..N), mean_gap = mean(proxy - eval), and the range each level spans over training. "
    "Descriptive: n <= 10 points per arm, no multiplicity correction. Proxy = training oracle by construction. "
    f"{CENSOR_F} Fixture: papers/2026_lookahead_pto_grpo/tables/reward_faithfulness_levels_rho.csv (exact)."))

# ── 3f · figures, one per eval grader ─────────────────────────────────────────────────────────
for j in JUDGES:
    name = "faithfulness" if j == PRIMARY_LABEL else ("faithfulness_heldout" if len(JUDGES) == 2 else f"faithfulness_{j}")
    with PAPER_RC:
        fig = plotting.faithfulness_fig(CURVE, MP, j)
    exports.save_fig(fig, name, caption=(
        f"**Training-reward faithfulness — eval side graded by {faithfulness.judge_display(j)}; proxy = the training "
        "oracle.** Panels (a) PTO and (b) GRPO: agreement between the proxy and the full-conversation eval Q1Q2 vs "
        "prefix length n_turns, pooled over iterations (PTO 1-10, GRPO 1-5 so the censored GRPO_LA5 is like-for-like); "
        "K=0 solid + circle, K=5 dashed + square, ribbons = 95% cluster-bootstrap CI over conversations, 0.5 = chance. "
        "Panel (c): the matched-policy (train_iter 1, same base policy in both K arms) K0 - K5 difference per bin with "
        f"CI; NEGATIVE = look-ahead more faithful. {CENSOR_F} Tables: `faithfulness_curve"
        + ("" if j == PRIMARY_LABEL else "_heldout") + "`, `faithfulness_matched_policy`."))
    plt.show()

## 4 · The tail audit — what the K=5 reward actually scored  `[TRAINING]`

**Purpose.** Under K=5 the oracle never sees `prefix + completion` alone: five simulated turns (patient, policy,
patient, policy, patient) are appended first, and the patient simulator may close the session inside them. So the
K=5 reward is a function of the tail as much as of the candidate. This section reads every tail the K=5 arms
recorded (`lookahead.tail` in `generations.jsonl`; K=0 arms have no tails) and asks: how do the tails end
(`tail_audit_by_iter`); what does the therapist say inside them vs in the candidate itself
(`tail_cues_by_iter`); does the tail's ending predict the reward *within the group the update sees*
(`tail_within_group`); and how does the within-group reward vary with realized tail length
(`tail_score_by_realized_turns`).

**Facts the audit rests on.** `realized_turns` = number of `[ROLE]:` labels in the tail (0..5); `ended_early` =
`realized_turns < 5`; odd counts end on a PATIENT turn, even (>0) on a THERAPIST turn. The literal `SESSION ENDED`
marker never appears in a tail (`handle_session_end` strips it) — what survives is a *fingerprint* (the kept text
ends in whitespace), validated per row by `wrapup_cue_rate_patient_closed` vs `wrapup_cue_rate_full_open`.
PTO candidates with an EMPTY completion carry no tail and no score (never simulated — dropped, and they cost no
calls); PTO_LA5 iteration 5 also has candidates with a tail but no score (an oracle-API incident). GRPO records
carry a TRL `eval`-phase block (scored, no gradient) — dropped here. **Log coverage (GRPO only):** a
crashed-and-resumed iteration logged only its post-resume steps (GRPO_LA5 iters 1–2 ≈ 0.5 / 0.7), so those rows
describe the later part of the iteration; PTO logs are complete.

**Sign.** `delta_ee_minus_full`: **+ ⇒ the ended-early candidates score HIGHER** than the full-tail candidates
of the same group (a within-arm, within-group contrast — the package-wide K convention does not apply). Grader:
the training oracle (not judge-swappable). GRPO_LA5 is censored at iteration 5. The API-call accounting from the
same generator (oracle / patient calls per iteration, K5/K0 ratios) belongs to the compute axis and renders in
`compute/cost`.

In [ ]:
AUD = tails.tail_audit_frames(KA)                     # ~1-2 min per K=5 arm; clears the generations memo per arm
TA_BYIT, TA_CUES, TA_WG, TA_RT = AUD.by_iter, AUD.cues, AUD.within_group, AUD.score_by_realized_turns
print(f"rows: " + "; ".join(f"{a} {info}" for a, info in AUD.rows.items()) + f"   [{time.time() - _T0:.0f} s]")
if AUD.scout_check:
    sc = AUD.scout_check
    print(f"[check] scout GRPO_LA5 iter 5, first 300 groups: ended_early {sc.get('ended_early')}/{sc.get('n')} "
          f"(fixture {tails.SCOUT_EXPECTED['ended_early']}/{tails.SCOUT_EXPECTED['n']}); rt {sc.get('realized_turns')}")
    assert sc.get("ended_early") == tails.SCOUT_EXPECTED["ended_early"] and sc.get("n") == tails.SCOUT_EXPECTED["n"], "scout anchor moved"

TAIL_ARMS = [a for a in ("PTO_LA5", "GRPO_LA5") if a in set(TA_BYIT.arm)] if len(TA_BYIT) else []
if not TAIL_ARMS:
    print("no K>0 arm with recorded tails — the tail audit is skipped.")
else:
    display(TA_BYIT[["arm", "train_iter", "n_groups", "n_candidates", "log_coverage", "realized_turns_mean", "ended_early_rate",
                     "patient_closed_share", "therapist_stalled_share", "no_tail_share", "full_share",
                     "wrapup_cue_rate_patient_closed", "wrapup_cue_rate_full_open"]].round(3))
    exports.save_table(TA_BYIT, "tail_audit_by_iter", caption=(
        "**Look-ahead TAIL structure per K=5 arm x training iteration** (+ a `pooled` row per arm). train_iter n = "
        "the branching done by policy pi_n, whose eval convs are model_iter_{n-1}; there is no tail at iteration 0 "
        "because look-ahead only runs during training. One row per candidate scored under K=5 (GRPO 'eval'-phase "
        "groups and PTO empty completions excluded); realized_turns = number of simulated turns in the tail (0..5), "
        "ended_early = realized_turns < 5. `rtK_share` = share of candidates with K realized turns; the end-reason "
        "shares classify how the tail stopped: `patient_closed` = the patient wrote SESSION ENDED (the marker is "
        "stripped by handle_session_end; identified by the trailing-whitespace fingerprint), `therapist_stalled` = an "
        "empty/degenerate therapist turn froze the sim, `after_therapist` = ended on a therapist turn, `no_tail` = "
        "zero simulated turns; `full_closed_at_turn5` = the fingerprint on a full 5-turn tail. `wrapup_cue_rate_*` "
        "validates the fingerprint: share of tails whose LAST patient turn carries a wrap-up phrase (wrap / for now / "
        "next time / thank ...) among patient_closed tails vs among full tails that did not end in whitespace. "
        "`tail_loop_rate` = a therapist turn in the tail verbatim-repeats the candidate or another tail turn. "
        "Bootstrap 95% CI over candidates (1,000 draws, seed = BOOT_SEED). `log_coverage` = logged groups / (optimizer "
        "steps x 16) for GRPO — iterations that crashed and resumed logged only their post-resume steps (GRPO_LA5 "
        "iters 1-2 ~0.5/0.7), so those rows describe the later half of the iteration; PTO logs are complete (1.0). "
        "`n_unscored_dropped` = candidates with a tail but no oracle score (an oracle-API incident at PTO_LA5 "
        f"iteration 5); `n_not_simulated_dropped` = PTO empty completions. {GROUP_UNIT} {CENSOR} Grader: the "
        f"{TRAIN_GRADER}. Fixture: papers/2026_lookahead_pto_grpo/tables/tail_audit_by_iter.csv (rates / counts "
        "exact; the ended_early CI re-drawn)."))

    display(TA_CUES.round(3))
    exports.save_table(TA_CUES, "tail_cues_by_iter", caption=(
        "**Simple lexical cues in the look-ahead tail's THERAPIST turns versus in the scored candidate itself**, per "
        "K=5 arm x train_iter (+ pooled): chars per turn, question marks per turn, the RE_AFFIRM / RE_EFFUSIVE "
        "regexes from eda_analysis.constants (directional sanity cues, not primary metrics), `tail_loop_rate`, "
        f"patient turn length. {GROUP_UNIT} {CENSOR} Grader: the {TRAIN_GRADER}. Fixture: "
        "papers/2026_lookahead_pto_grpo/tables/tail_audit_cues_by_iter.csv (exact)."))

    display(TA_WG[["arm", "train_iter", "n_groups", "n_groups_both", "delta_ee_minus_full", "dz", "ci_lo", "ci_hi", "p", "p_holm",
                   "base_ee_rate", "p_chosen_is_ee", "p_chosen_given_ee", "p_chosen_given_full", "rr_chosen_ee_vs_full"]].round(3))
    exports.save_table(TA_WG, "tail_within_group", caption=(
        "**Does the tail predict the reward WITHIN a group** (the unit the update sees: GRPO's G=8 siblings, PTO's M=8 "
        "branches)? `rho_dev_vs_*` = Spearman of the group-demeaned score against the group-demeaned realized_turns / "
        "tail chars (pooled over groups with within-group variance). `delta_ee_minus_full` = mean(score | ended early) "
        "- mean(score | full 5-turn tail) paired within groups holding both kinds (SIGN: **+ => the ended-early "
        "candidates score HIGHER**); dz, bootstrap 95% CI (seed = BOOT_SEED), Wilcoxon p, `p_holm` = Holm within arm "
        "across iterations (the pooled row is not part of the family). `z_*` = the group-standardised score (GRPO's "
        "advantage). `p_chosen_is_ee` = share of groups whose argmax candidate ended early vs `base_ee_rate` = share "
        "of all candidates that ended early (`chosen_minus_base` + group-bootstrap CI); `p_chosen_given_ee` / `_full` "
        "= per-candidate P(argmax) by tail kind (1/8 = chance) and their ratio `rr_chosen_ee_vs_full` with a "
        "group-bootstrap 95% CI. `log_coverage` as in `tail_audit_by_iter` (GRPO_LA5 iters 1-2 are partial logs). "
        f"{GROUP_UNIT} {CENSOR} Grader: the {TRAIN_GRADER}. Fixture: "
        "papers/2026_lookahead_pto_grpo/tables/tail_audit_within_group.csv (deltas / dz / p / rho / rates exact; "
        "CI bounds re-drawn)."))

    display(TA_RT[TA_RT.train_iter == "pooled"].round(3))
    exports.save_table(TA_RT, "tail_score_by_realized_turns", caption=(
        "**Mean within-group score deviation (score - group mean; Q1Q2 points) and P(argmax) by realized_turns**, per "
        "K=5 arm and train_iter (+ pooled); bootstrap 95% CI over candidates (500 draws, seed = BOOT_SEED), cells with "
        ">= 5 candidates only. Odd realized_turns = the tail ends on a patient turn (the patient closed the session), "
        f"even = ends on a therapist turn (0 = no tail). {GROUP_UNIT} {CENSOR} Grader: the {TRAIN_GRADER}. Fixture: "
        "papers/2026_lookahead_pto_grpo/tables/tail_audit_score_by_realized_turns.csv (means / counts / P(argmax) "
        "exact; CIs re-drawn)."))

    with PAPER_RC:
        fig = plotting.tail_audit_fig(TA_BYIT, TA_RT, TA_WG, arms=TAIL_ARMS, palette=S.PALETTE)
    if fig is not None:
        exports.save_fig(fig, "tail_audit", caption=(
            "**Look-ahead tails of the K=5 arms.** (a) share of candidates whose 5-turn tail ended early, by training "
            "iteration (with the patient-closed share dotted underneath — almost every early ending is the patient "
            "closing the session), 95% bootstrap CI; (b) within-group reward (score - group mean, Q1Q2 points) by "
            "realized tail length, pooled over iterations — odd = patient closed, even = ended on a therapist turn; "
            "(c) P(candidate is the group argmax) for ended-early vs full-tail candidates by iteration, 1/8 = chance. "
            f"PTO_LA5 dashed + square, GRPO_LA5 as drawn in the legend; colour = arm. Grader: the {TRAIN_GRADER}. "
            f"{GROUP_UNIT} {CENSOR} Tables: `tail_audit_by_iter`, `tail_score_by_realized_turns`, `tail_within_group`."))
        plt.show()

## 5 · Ledgers  `[TRAINING] → [EVAL]`

Every quotable cell of the §2–§4 tables, as `results/lookahead/mechanism/tables/{dispersion,faithfulness,tails}_numbers.json`
(`{dotted.key: {value, source, note}}`, the same key families as the paper's frozen `analysis/out/{dispersion_by_k,
reward_faithfulness,tail_audit}.json`). A paper's `NUMBERS.md` cites `<ledger>.json :: <key>` rather than
re-typing a number from a table. `tails_numbers` also carries the API-call keys (`api.*`, `api_ratio.*`,
`api_totals.*`) whose TABLES render in `compute/cost` — the module ledger is one object, so it is saved whole here.

In [ ]:
NUM_D = dispersion.dispersion_numbers(BYIT, RAT, TAU, EXP, extra_meta={
    "arms": KA_LABELS, "n_candidates_loaded": int(len(GENS)), "n_groups": int(len(G)), "seed": BOOT_SEED})
p = exports.save_numbers("dispersion_numbers", NUM_D, caption=(
    "**Number ledger for the dispersion test (section 2)** — every cell of `dispersion_by_iter` (by_iter.*), "
    "`dispersion_ratios` (ratios.*, headline.*), `dispersion_tau` (tau.*) and `dispersion_expectation` "
    "(expectation.*), plus the GRPO group_std check, as `{value, source, note}` records keyed like the paper's "
    f"dispersion_by_k.json. Ratios are K5/K0 (> 1 = K=5 wider). {GROUP_UNIT} {CENSOR} Grader: the {TRAIN_GRADER}."))
print(f"dispersion: {len(NUM_D)} keys -> {os.path.relpath(p, S.RESULTS_DIR)}")

NUM_F = faithfulness.faithfulness_numbers(FD, curve=CURVE, by_iter=BYITER, k_by_iter=DK, matched=MP, tests=TESTS,
                                          summary=KSUM, coop=COOP, levels=LV, rho=RHO, selfcheck_max_dev=MAX_DEV)
p = exports.save_numbers("faithfulness_numbers", NUM_F, caption=(
    "**Number ledger for reward faithfulness (section 3)** — headline bins of `faithfulness_curve_long` (curve.*), "
    "`faithfulness_k_by_iter` (k_by_iter.*), `faithfulness_curve_by_iter` (by_iter.*), `faithfulness_k_summary` "
    "(k_summary.*), `faithfulness_matched_policy[_tests]` (matched_policy*.*), `faithfulness_by_coop` incl. the "
    "within-stratum K deltas (by_coop.*), `faithfulness_levels[_rho]` (levels*.*), the bootstrap spec, the caveats and "
    "the self-check against stats.rank_agreement_by_nturns, as `{value, source, note}` records keyed like the paper's "
    f"reward_faithfulness.json (grader keys primary / heldout). {SIGN} {GRADERS} {CENSOR_F}"))
print(f"faithfulness: {len(NUM_F)} keys -> {os.path.relpath(p, S.RESULTS_DIR)}")

NUM_T = tails.tails_numbers(KA, audit=AUD)              # also builds the API-call frames (~2 min) — their tables live in compute/cost
p = exports.save_numbers("tails_numbers", NUM_T, caption=(
    "**Number ledger for the tail audit (section 4)** — row-filter counts (rows.*), the scout anchor "
    "(scout_check.*: GRPO_LA5 iter 5, first 300 groups), every cell of `tail_audit_by_iter` (by_iter.*), "
    "`tail_within_group` (within_group.*), `tail_cues_by_iter` (cues.*) and the pooled rows of "
    "`tail_score_by_realized_turns` (score_by_realized_turns.*), plus the API-call accounting keys (api.*, "
    "api_ratio.*, api_totals.*) whose tables render in compute/cost, as `{value, source, note}` records keyed like "
    "the paper's tail_audit.json. Sign of within_group deltas: + => ended-early candidates score higher. "
    f"{GROUP_UNIT} {CENSOR} Grader: the {TRAIN_GRADER}."))
print(f"tails: {len(NUM_T)} keys -> {os.path.relpath(p, S.RESULTS_DIR)}   [{time.time() - _T0:.0f} s total]")

## 6 · How to read this family
- **The chain (§1) is read top-down.** If the first gate (what the reward selects for) already differs by K, the
  look-ahead lever acts on the *reward*; if only the outcome differs, the pool and the eval are moving for another
  reason. Rows 1–2 are what the training oracle chose during the run and no re-grading can change them.
- **Dispersion (§2) is a null test, not a result.** `ratio_of_ratios` ≈ 1 with `margin_over_sd` and `winner_z`
  at the iid reference means look-ahead *rescales* the within-group spread; only a `winner_z_diff` CI above 0 would
  say the winner stands out more. `share_gap_closed_r1` is how much of K=5's higher PTO pair yield the τ filter
  manufactures from that rescaling. Ratios here are K5/K0 — the caption of every table says so.
- **Faithfulness (§3) has one clean row: `train_iter 1`.** It is the only place both K arms branch from the same
  base policy, so K0 − K5 there is the measurement effect alone; every later row compares diverged policies. Sign
  is + ⇒ K=0 higher, so a NEGATIVE delta means look-ahead is the more faithful proxy. The held-out grader changes
  only the EVAL side of the join — the proxy is the training oracle by construction.
- **Tails (§4) are the reward's blind spot under K=5.** An early-ending tail is a real event (the patient closed),
  and `tail_within_group` says whether the update rewards it *within the group it sees* (+ ⇒ ended-early scores
  higher). Log coverage on GRPO_LA5 iterations 1–2 is partial (crash + resume); PTO logs are complete.
- **Pairing units differ by section** — the group in §1–§2 and §4, the conversation (cluster bootstrap) in §3 —
  and every CI here is re-drawn under `constants.BOOT_SEED` (the paper fixture used its own seeds), so CI bounds
  may differ from the paper in the third decimal while every point estimate is exact.
- **Censoring.** GRPO K=5 stops at iteration 5 (its full budget). Iteration ≠ spend — the budget-matched reading
  of the same lever is `compute/cost`, which also carries this generator's API-call tables.
- _(The measured values are narrated in `results/lookahead/SUMMARY.md`; the caveats they imply in
  `results/LIMITATIONS.md`. This notebook is where they are computed.)_

In [ ]:
exports.prune_orphan_captions(); print("index ->", exports.build_index())